# Linear Transformer for Blood Pressure Estimation

This notebook implements a Linear Transformer model for blood pressure estimation from pulse waveforms, with attention masks initialized from a pretrained LodeSTAR model.

In [34]:
import os
import sys
import getpass
user = getpass.getuser()
os.environ["POLARS_VERBOSE"] = "1"
import numpy as np
import pandas as pd
import json
# Include path to pyTPBI

if not f'/home/{user}/pyTBPI/' in sys.path:
    sys.path.append(f'/home/{user}/pyTBPI/') 
if not f'/home/{user}/Utilities/' in sys.path:
    sys.path.append(f'/home/{user}/Utilities/')
if not f'/home/{user}/Utilities/dynutilities' in sys.path:
    sys.path.append(f'/home/{user}/Utilities/dynutilities')  


from ViTrackModules import ProcessingUtilities as pu
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm import tqdm
import polars as pl
from functools import lru_cache
import wandb  # Weights & Biases for experiment tracking
from sklearn.metrics import mean_squared_error
from LinearTransformerModel import LinearTransformer

os.environ["POLARS_MAX_THREADS"] = "1" 

%load_ext autoreload
%autoreload 2   


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [35]:
class Config:
    # Data parameters
    seq_len =120
    batch_size = 8
    num_workers = 4
    
    # Model parameters
    input_dim = 2  # Assuming input has 2 channels (pulse and motion)
    embed_dim = 128
    num_heads = 8
    num_layers = 6
    hidden_dim = 512
    dropout = 0.1
    
    # Training parameters
    lr = 1e-4
    weight_decay = 1e-5
    epochs = 100
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Paths
    lodestar_weights_path = 'path_to_lodestar_weights.pth'
    checkpoint_dir = 'checkpoints'
    
    # Weights for loss components
    bp_loss_weight = 1.0
    pulse_loss_weight = 0.5
    
    # Initialize Weights & Biases
    use_wandb = True
    project_name = "bp_estimation_transformer"
    
config = Config()

# Create checkpoint directory
os.makedirs(config.checkpoint_dir, exist_ok=True)

'''Read controids trajectories from path to a numpy array

Input : str path to load

Output : np.array Dimension: [segmentLength, numCentroids, 2]  
'''
def read_from_csv(centroids_traj_fname):

    cents_pos = pd.read_csv(centroids_traj_fname)

    if 'Unnamed: 0' in cents_pos.columns:
        centroids = cents_pos.drop('Unnamed: 0', axis=1)
    else: 
        centroids  = cents_pos
    centroidSnaps     = centroids.values
    odd_columns       = centroidSnaps[:, 1::2]
    even_columns      = centroidSnaps[:, ::2]
    if odd_columns.shape != even_columns.shape:
        min_cols = min(odd_columns.shape[1], even_columns.shape[1])
        odd_columns  = odd_columns[:, :min_cols]
        even_columns = even_columns[:, :min_cols]
    centroid_trajectories = np.stack([odd_columns, even_columns], axis=2)   
    return centroid_trajectories

In [36]:
DataSegments = pd.read_csv(os.path.expanduser('~/DynoNAS/Peter/Research_data/DataSegments_DotTrackingAndMotionCorrection.csv'))
display(DataSegments.head())

sid = '2024y_03m_14d_12h_14m_19s_787ms_358us_tracking_0'
            
cond1 = DataSegments['unique_file_id'] == sid.partition('us')[0] + sid.partition('us')[1]
cond2  = DataSegments['relative_idx'] == int(sid[-1])
cond3 = DataSegments['type'] == 'tracking'
cond4 = DataSegments['Artifact'] == 'no'
subject_id = DataSegments[  cond3 & cond4]
display(subject_id)

,Unnamed: 0,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,...,end_idx,relative_idx,schema_version,phantom data,Artifact,BP changes,notes,class,processed_data_path,centroids_trajectories_data_path
0,0,NaN,0,NaN,DynoNAS/jordan/ContTrack/InducedArtifact/Victo...,2024y_10m_24d_14h_00m_35s_052ms_674us,NaN,NaN,NaN,induced,...,19300.0,0.0,NaN,no,yes,no,"continuous motion in sequence: finger motion, ...",Continuous Motion Sequences,/home/peter/DynoNAS/Peter/Research_data/induce...,/home/peter/DynoNAS/Peter/Research_data/induce...
1,1,NaN,0,NaN,DynoNAS/jordan/ContTrack/InducedArtifact/2024y...,2024y_10m_29d_14h_14m_58s_487ms_881us,NaN,NaN,NaN,induced,...,18001.0,0.0,NaN,no,yes,no,"continuous motion in sequence: finger motion, ...",Continuous Motion Sequences,/home/peter/DynoNAS/Peter/Research_data/induce...,/home/peter/DynoNAS/Peter/Research_data/induce...
2,2,NaN,0,NaN,DynoNAS/jordan/ContTrack/InducedArtifact/2024y...,2024y_10m_28d_15h_32m_02s_947ms_513us,NaN,NaN,NaN,induced,...,27093.0,0.0,NaN,no,yes,no,"continuous motion in sequence: finger motion, ...",Continuous Motion Sequences,/home/peter/DynoNAS/Peter/Research_data/induce...,/home/peter/DynoNAS/Peter/Research_data/induce...
3,3,NaN,0,NaN,DynoNAS/jordan/ContTrack/InducedArtifact/Valsa...,2024y_11m_08d_12h_39m_16s_831ms_632us,NaN,NaN,NaN,induced,...,8000.0,0.0,NaN,no,yes,yes,val slava,Named Physiological Maneuvers,/home/peter/DynoNAS/Peter/Research_data/induce...,/home/peter/DynoNAS/Peter/Research_data/induce...
4,4,NaN,0,NaN,DynoNAS/jordan/ContTrack/InducedArtifact/Valsa...,2024y_11m_08d_12h_39m_16s_831ms_632us,NaN,NaN,NaN,induced,...,15000.0,0.0,NaN,no,yes,yes,val slava,Named Physiological Maneuvers,/home/peter/DynoNAS/Peter/Research_data/induce...,/home/peter/DynoNAS/Peter/Research_data/induce...


,Unnamed: 0,id,subject_id,record_id,nas_path,unique_file_id,visit_id,visit_type,hospital_name,study_name,...,end_idx,relative_idx,schema_version,phantom data,Artifact,BP changes,notes,class,processed_data_path,centroids_trajectories_data_path
158,158,5167.0,20250428,1054.0,DynoNAS/Research_data/GeneralTesting/2025y_04m...,2025y_04m_28d_15h_42m_24s_956ms_770us,312.0,NaN,DynoBoston,NaN,...,19756.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202504...,/home/peter/DynoNAS/Peter/Research_data/202504...
159,159,5168.0,20250428,1055.0,DynoNAS/Research_data/GeneralTesting/2025y_04m...,2025y_04m_28d_16h_23m_12s_412ms_267us,312.0,NaN,DynoBoston,NaN,...,17653.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202507...,/home/peter/DynoNAS/Peter/Research_data/202507...
160,160,5169.0,20250715,1065.0,DynoNAS/Research_data/GeneralTesting/2025y_07m...,2025y_07m_15d_11h_09m_59s_710ms_889us,314.0,NaN,DynoBoston,NaN,...,11923.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
161,161,5227.0,20250529,1073.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_11h_11m_10s_919ms_329us,319.0,NaN,DynoBoston,NaN,...,25356.0,1.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
162,162,5228.0,20250529,1073.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_11h_11m_10s_919ms_329us,319.0,NaN,DynoBoston,NaN,...,27391.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
163,163,5234.0,20250529,1074.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_10h_36m_21s_029ms_156us,319.0,NaN,DynoBoston,NaN,...,34918.0,1.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
164,164,5235.0,20250529,1074.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_10h_36m_21s_029ms_156us,319.0,NaN,DynoBoston,NaN,...,15697.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
165,165,5246.0,20250529,1076.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_14h_42m_03s_448ms_111us,319.0,NaN,DynoBoston,NaN,...,21491.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/202505...,/home/peter/DynoNAS/Peter/Research_data/202505...
166,166,5250.0,20250529,1077.0,DynoNAS/Research_data/GeneralTesting/2025y_05m...,2025y_05m_29d_17h_53m_00s_440ms_287us,319.0,NaN,DynoBoston,NaN,...,36817.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/6_2024...,/home/peter/DynoNAS/Peter/Research_data/6_2024...
167,167,627.0,6,146.0,DynoNAS/Clinical_Studies/ClevelandClinic/CCF_D...,2024y_03m_14d_12h_14m_19s_787ms_358us,60.0,in_patient,ClevelandClinic,NaN,...,54256.0,0.0,3.0,False,no,no,clean,clean,/home/peter/DynoNAS/Peter/Research_data/6_2024...,/home/peter/DynoNAS/Peter/Research_data/6_2024...


#### Covnert .csv files in Synthetic_Videos_Noise_Mixture_Protocol folder to .parquet for faster loading


In [40]:
from tqdm import tqdm

def preload_and_convert(centroids_dir):
    files = [f for f in os.listdir(centroids_dir) if f.endswith(".csv")]
    print(f"Preparing {len(files)} files...")
    
    for f in tqdm(files):
        csv_path = os.path.join(centroids_dir, f)
        parquet_path = csv_path.replace(".csv", ".parquet")
        
        if not os.path.exists(parquet_path):
            # Use 'infer_schema=False' to treat EVERYTHING as a string initially.
            # This is the fastest way to avoid the "stuck" inference phase.
            pl.read_csv(csv_path, infer_schema=False).write_parquet(parquet_path)

# Run this once BEFORE you create your Dataset object

syntheszied_video_data_path= os.path.expanduser('~/DynoNAS/Peter/Research_data/Synthetic_Videos_Noise_Mixture_Protocol')
preload_and_convert(syntheszied_video_data_path)

Preparing 996 files...


  0%|          | 0/996 [00:00<?, ?it/s]_init_credential_provider_builder(): credential_provider_init = None
polars-stream: updating graph state
async thread count: 4
polars-stream: running in-memory-source in subgraph
polars-stream: running io-sink[single-file[parquet]] in subgraph
async upload_chunk_size: 67108864
io-sink[single-file[parquet]]: start_single_file_sink_pipeline: file_writer_starter: parquet, takeable_rows_provider: TakeableRowsProvider { max_size: NonZeroRowCountAndSize { num_rows: 122880, num_bytes: 18446744073709551615 }, byte_size_min_rows: 16384, allow_non_max_size: false }, upload_chunk_size: 67108864
polars-stream: done running graph phase
polars-stream: updating graph state
io-sink[single-file[parquet]]: Join on task_handle (recv PortState::Done)
io-sink[single-file[parquet]]: Statistics: total_size: RowCountAndSize { num_rows: 49, num_bytes: 35452 }
_init_credential_provider_builder(): credential_provider_init = None
polars-stream: updating graph state
polars-st

## Dataset and DataLoader

In [48]:
class BPDataset(Dataset):
    """
    @brief  This dataset loads dot-tracking based centroids waveforms as data, and blood pressure data and clean IAH as labels.
    The Linear Transformer model uses centroid waveforms to predict blood pressure and clean IAH.

    @param points_data_path: Path to the directory containing the centroid waveforms.
    @param BP_data_path: Path to the directory containing the blood pressure data.
    @param DataSegments: Polars DataFrame containing the data segments.
    @param window_size: Size of the window to use for the data.
    @param overlap: Overlap between windows.
    @param use_overlap: Whether to use overlap.
    @param transform: Transform to apply to the data.

    @return: A dataset object.
    """
    def __init__(
        self,
        points_data_path,
        BP_data_path,
        DataSegments,              # Polars DataFrame recommended
        window_size=120,
        overlap=20,
        use_overlap=True,
        transform=None,
    ):
        self.centroids_dir = points_data_path
        self.bp_dir = BP_data_path
        self.window_size = int(window_size)
        self.overlap = int(overlap)
        self.use_overlap = use_overlap
        self.transform = transform
        self.stride = self.window_size - self.overlap if use_overlap else self.window_size

        # ------------------------------------------------------------------
        # Identify subjects
        # ------------------------------------------------------------------
        files = os.listdir(self.centroids_dir)
        self.subject_ids = sorted(
            f.split("_r0.05_CentroidPositionsLowPass.csv")[0]
            for f in files
            if f.endswith("_r0.05_CentroidPositionsLowPass.csv")
        )

        # ------------------------------------------------------------------
        # Build window index (lightweight metadata only)
        # ------------------------------------------------------------------
        self.index = []

        for sid in self.subject_ids:
            try:
                clean_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.05_CentroidPositionsLowPass.csv"
                )
                easy_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.35_CentroidPositionsLowPass.csv"
                )
                hard_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.65_CentroidPositionsLowPass.csv"
                )
                noise_path = os.path.join(
                    self.centroids_dir, f"{sid}_r0.95_CentroidPositionsLowPass.csv"
                )

                unique_file_id = sid.split('_tracking_')[0]
                relative_idx = int(sid.split('_tracking_')[-1])

                subject_segment = DataSegments[
                    (DataSegments["unique_file_id"] == unique_file_id) &
                    (DataSegments["relative_idx"] == relative_idx) &
                    (DataSegments["type"] == "tracking")
                ]

                subject_id = subject_segment['subject_id'].iloc[0]

                bp_path = subject_segment['nas_path'].iloc[0]

                config_json_path = os.path.join(
                    self.bp_dir, f"{subject_id}_{sid}", f"{sid}_config.json"
                )

                iah_path = json.load(open(config_json_path))['wave_path'] # Path to clean pulse waveform

                # Determine minimum synchronized length without loading full arrays
                def csv_len(path):
                    df = pl.read_csv(path, columns=[0])
                    return df.height

                min_len = min([
                    csv_len(clean_path),
                    csv_len(easy_path),
                    csv_len(hard_path),
                    csv_len(noise_path),
                ])
                # print('min_len - self.window_size + 1:', min_len - self.window_size + 1)
                # print('self.stride:', self.stride)
                for start in range(0, min_len - self.window_size + 1, self.stride):
                    # print('start:', start)
                    self.index.append(
                        (sid, start, bp_path, iah_path)
                    )
                print(len(self.index))

            except Exception as e:
                print(f"Skipping subject {sid}: {e}")

    # ----------------------------------------------------------------------
    # Cached loaders (per worker)
    # ----------------------------------------------------------------------
    @lru_cache(maxsize=32)
    def _load_centroid_csv(self, path, start, length):

        parquet_path = self._ensure_parquet(path)

        flat_centroid_array = pl.scan_parquet(parquet_path).slice(start, length).collect().to_numpy()

        centroid_x = flat_centroid_array[:, ::2] # odd columns
        centroid_y = flat_centroid_array[:, 1::2] # even columns

        return np.concatenate([centroid_x, centroid_y], axis=2)

    @lru_cache(maxsize=32)
    def _load_bp(self, bp_path, start, length):

        # parquet_path = self._ensure_parquet(bp_path)

        # IAH = pl.scan_parquet(parquet_path).slice(start, end).select("PULSE_Y").to_numpy().squeeze()

        # IAP = 0.32 * pl.scan_parquet(parquet_path).select("PRS_IAP").to_numpy().squeeze()[start:end]

        IAH = pl.read_csv(bp_path).slice(start, length).select("PULSE_Y").to_numpy().squeeze()

        IAP = 0.32 * pl.read_csv(bp_path).slice(start, length).select("PRS_IAP").to_numpy().squeeze()

        BP,_,_ = pu.linearCalibrate1DtoBp(IAH, IAP, FPS=30, winsizeSec=10, cwinstart=0) # Map IAH to BP

        return BP

    @lru_cache(maxsize=32)
    def _load_clean_pulse_waveform(self, iah_path, start, length):  
        # parquet_path = self._ensure_parquet(iah_path)
        # return pl.scan_parquet(parquet_path).slice(start, end).select("IAH_y").to_numpy().squeeze()

        return pl.read_csv(iah_path).slice(start, end).select("PULSE_Y").to_numpy().squeeze()
    # ----------------------------------------------------------------------
    def __len__(self):
        return len(self.index)

    # ----------------------------------------------------------------------
    def __getitem__(self, idx):
        sid, start, bp_path, iah_path = self.index[idx]
        end = start + self.window_size

        clean = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.05_CentroidPositionsLowPass.csv")
        , start, self.window_size)

        easy = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.35_CentroidPositionsLowPass.csv")
        , start, self.window_size)

        hard = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.65_CentroidPositionsLowPass.csv")
        ,start, self.window_size)

        noise = self._load_centroid_csv(
            os.path.join(self.centroids_dir, f"{sid}_r0.95_CentroidPositionsLowPass.csv")
        ,start, self.window_size)

        bp = self._load_bp(bp_path, start, self.window_size)

        clean_pulse = self._load_clean_pulse_waveform(iah_path, start, self.window_size)


        if self.transform:
            clean_centroids = self.transform(clean)
            easy_centroids = self.transform(easy)
            hard_centroids = self.transform(hard)
            noise_centroids = self.transform(noise)

        return (
            torch.from_numpy(clean_centroids).float(),
            torch.from_numpy(easy_centroids).float(),
            torch.from_numpy(hard_centroids).float(),
            torch.from_numpy(noise_centroids).float(),
            torch.from_numpy(bp).float(),
            torch.from_numpy(clean_pulse).float(),
        )
    
    def _ensure_parquet(self, csv_path):

        """Converts CSV to Parquet if it doesn't exist. Returns the parquet path."""
        parquet_path = csv_path.replace(".csv", ".parquet")
        
        if not os.path.exists(parquet_path):
            print(f"Converting {os.path.basename(csv_path)} to Parquet...")
            try:
                # 1. Use read_csv with explicit dtypes to skip the "sourcing schema" hang
                # 2. Use write_parquet (In-memory) instead of sink_parquet (Streaming)
                # Note: Adjust dtypes based on your Centroid CSV columns
                df = pl.read_csv(
                    csv_path, 
                    infer_schema_length=0, # Skip automatic scanning
                )
                # sink_parquet uses streaming for memory efficiency
                pl.scan_csv(csv_path).write_parquet(parquet_path)
            except Exception as e:
                print(f"Failed to convert {csv_path}: {e}")
                return csv_path  # Fallback to CSV if conversion fails
                
        return parquet_path


In [49]:
   
# Create datasets and dataloaders
full_dataset = BPDataset(points_data_path= os.path.expanduser('~/DynoNAS/Peter/Research_data/Synthetic_Videos_Noise_Mixture_Protocol'),
                        BP_data_path= os.path.expanduser('~/DynoNAS/Peter/Research_data'),
                        DataSegments=DataSegments,
                        window_size= 45,
                        overlap=30,
                        use_overlap=True,
                        transform=None)  
print("Dataset length:", len(full_dataset))

num_cpus = os.cpu_count() or 1

# 2. Split the dataset (0.7 Train, 0.3 Test)
train_size = int(0.7 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset,
                          batch_size=8,
                          shuffle=True,
                          num_workers=1, # min(4, num_cpus), # Conservative start
                          pin_memory=True,              # Faster data transfer to GPU
                          persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

test_loader = DataLoader(test_dataset,
                         batch_size=8,
                         shuffle=False,
                         num_workers=1, # min(4, num_cpus), # Conservative start
                         pin_memory=True,              # Faster data transfer to GPU
                         persistent_workers=True       # Keeps workers alive between epochs for speed)
                        )

51
109
157
208
230
261
292
323
355
387
417
437
469
499
531
563
594
625
656
687
720
755
789
823
855
887
922
975
1026
Skipping subject 2024y_07m_31d_12h_57m_29s_035ms_910us_tracking_0: No such file or directory (os error 2): /home/peter/DynoNAS/Peter/Research_data/Synthetic_Videos_Noise_Mixture_Protocol/2024y_07m_31d_12h_57m_29s_035ms_910us_tracking_0_r0.35_CentroidPositionsLowPass.csv
1055
1101
1143
1191
1238
1288
1336
1384
1434
1484
1538
1585
Dataset length: 1585


### Verify the output of the dataloader is correct

In [52]:
import time
import torch
from tqdm import tqdm

def test_dataloader(dataloader, num_batches=20):
    print(f"Starting test on {type(dataloader.dataset).__name__}...")
    
    # Check if dataset is empty
    if len(dataloader) == 0:
        print("Error: Dataset is empty.")
        return

    start_time = time.perf_counter()
    samples_processed = 0
    
    try:
        # Iterate through a limited number of batches
        for i, batch in enumerate(tqdm(dataloader, total=num_batches)):
            if i >= num_batches:
                break
            
            # 1. Test Data Integrity (Shapes and Types)
            # Batch items: clean, easy, hard, noise, bp, clean_pulse
            if i == 0:
                print("\n--- Batch Integrity Check ---")
                labels = ["Clean", "Easy", "Hard", "Noise", "BP", "Pulse"]
                for name, data in zip(labels, batch):
                    print(f"{name:5} | Shape: {list(data.shape)} | Type: {data.dtype}")
                print("-----------------------------\n")

            # 2. Check for NaNs (Common in CSV processing)
            for j, data in enumerate(batch):
                if torch.isnan(data).any():
                    print(f"Warning: NaN detected in batch {i}, element {j}")

            samples_processed += batch[0].size(0)

        end_time = time.perf_counter()
        total_time = end_time - start_time
        fps = samples_processed / total_time

        print(f"\nTest Completed Successfully!")
        print(f"Processed {samples_processed} samples in {total_time:.2f}s")
        print(f"Throughput: {fps:.2f} samples/sec")

    except Exception as e:
        print(f"\nTest Failed with error: {e}")
        # Hint for your specific loader: check if lru_cache is causing memory bloat
        # or if a specific CSV file is missing/corrupt.

# Run the test
test_dataloader(train_loader, num_batches=20)


# for batch in train_loader:
#     clean_centroids, easy_centroids, hard_centroids, noise_centroids, bp, clean_pulse = batch
#     print('clean_centroids.shape: ', clean_centroids.shape)
#     print('easy_centroids.shape: ', easy_centroids.shape)
#     print('hard_centroids.shape: ', hard_centroids.shape)
#     print('noise_centroids.shape: ', noise_centroids.shape)
#     print('bp.shape: ', bp.shape)
#     print('clean_pulse.shape: ', clean_pulse.shape)
    
#     print("--------------------------------")
#     print("Verify the BP range is sensible (60-140)")
#     print(bp.min(), bp.max())
#     print("verify the clean_pulse range is sensible (0-1)")
#     print(clean_pulse.min(), clean_pulse.max())

#     break



KeyboardInterrupt: 

## Model Initialization

In [7]:
def initialize_model():
    """Initialize the model and load pretrained attention masks if available"""
    model = LinearTransformer(
        input_dim=config.input_dim,
        embed_dim=config.embed_dim,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
        max_seq_len=config.seq_len
    )
    
    # Load pretrained attention masks from LodeSTAR if available
    if os.path.exists(config.lodestar_weights_path):
        try:
            model.load_pretrained_attention_masks(config.lodestar_weights_path)
            print("Successfully loaded attention masks from LodeSTAR model")
        except Exception as e:
            print(f"Error loading attention masks: {e}")
    
    return model.to(config.device)

model = initialize_model()

## Training Setup

In [6]:
def setup_training(model):
    """Set up optimizer, scheduler, and loss functions"""
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay
    )
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
    )
    
    # Loss functions
    bp_criterion = nn.MSELoss()  # For BP estimation
    pulse_criterion = nn.MSELoss()  # For pulse waveform prediction
    
    return optimizer, scheduler, bp_criterion, pulse_criterion

optimizer, scheduler, bp_criterion, pulse_criterion = setup_training(model)

NameError: name 'model' is not defined

## Training Loop

In [63]:
def train_epoch(model, dataloader, optimizer, bp_criterion, pulse_criterion, epoch):
    model.train()
    total_loss = 0.0
    bp_loss_total = 0.0
    pulse_loss_total = 0.0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for i, batch in enumerate(progress_bar):
        
        print("Fetched batch", i)
        break
        # Move data to device
        inputs = batch['input'].to(config.device)
        bp_targets = batch['bp'].to(config.device)
        pulse_targets = batch['pulse'].to(config.device)
        
        # Forward pass
        optimizer.zero_grad()
        bp_pred, pulse_pred = model(inputs)
        
        # Calculate losses
        bp_loss = bp_criterion(bp_pred, bp_targets)
        pulse_loss = pulse_criterion(pulse_pred, pulse_targets)
        
        # Combine losses
        loss = config.bp_loss_weight * bp_loss + config.pulse_loss_weight * pulse_loss
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Update metrics
        total_loss += loss.item()
        bp_loss_total += bp_loss.item()
        pulse_loss_total += pulse_loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': total_loss / (progress_bar.n + 1),
            'bp_loss': bp_loss_total / (progress_bar.n + 1),
            'pulse_loss': pulse_loss_total / (progress_bar.n + 1)
        })
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_bp_loss = bp_loss_total / len(dataloader)
    avg_pulse_loss = pulse_loss_total / len(dataloader)
    
    # Log average losses
    print(f"Epoch {epoch+1} [Train] - Avg Loss: {avg_loss:.4f}, Avg BP Loss: {avg_bp_loss:.4f}, Avg Pulse Loss: {avg_pulse_loss:.4f}")
    
    return avg_loss, avg_bp_loss, avg_pulse_loss


In [64]:
def validate_epoch(model, dataloader, bp_criterion, pulse_criterion, epoch):
    model.eval()
    total_loss = 0.0
    bp_loss_total = 0.0
    pulse_loss_total = 0.0
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Val]")
        
        for batch in progress_bar:
            # Move data to device
            inputs = batch['input'].to(config.device)
            bp_targets = batch['bp'].to(config.device)
            pulse_targets = batch['pulse'].to(config.device)
            
            # Forward pass
            bp_pred, pulse_pred = model(inputs)
            
            # Calculate losses
            bp_loss = bp_criterion(bp_pred, bp_targets)
            pulse_loss = pulse_criterion(pulse_pred, pulse_targets)
            
            # Combine losses
            loss = config.bp_loss_weight * bp_loss + config.pulse_loss_weight * pulse_loss
            
            # Update metrics
            total_loss += loss.item()
            bp_loss_total += bp_loss.item()
            pulse_loss_total += pulse_loss.item()
            
            # Update progress bar
            progress_bar.set_postfix({
                'val_loss': total_loss / (progress_bar.n + 1),
                'val_bp_loss': bp_loss_total / (progress_bar.n + 1),
                'val_pulse_loss': pulse_loss_total / (progress_bar.n + 1)
            })
    
    # Calculate average losses
    avg_loss = total_loss / len(dataloader)
    avg_bp_loss = bp_loss_total / len(dataloader)
    avg_pulse_loss = pulse_loss_total / len(dataloader)
    
    # Log average losses
    print(f"Epoch {epoch+1} [Val] - Avg Loss: {avg_loss:.4f}, Avg BP Loss: {avg_bp_loss:.4f}, Avg Pulse Loss: {avg_pulse_loss:.4f}")
    
    return avg_loss, avg_bp_loss, avg_pulse_loss

In [65]:
# Training Execution
def main():
    # Initialize Weights & Biases if enabled
    if config.use_wandb:
        wandb.init(project=config.project_name, config=vars(config))
        wandb.watch(model)
    
    # Training loop
    best_val_loss = float('inf')
    
    for epoch in range(config.epochs):
        # Train for one epoch
        train_loss, train_bp_loss, train_pulse_loss = train_epoch(
            model, train_loader, optimizer, bp_criterion, pulse_criterion, epoch
        )
        
        # Validate
        val_loss, val_bp_loss, val_pulse_loss = validate_epoch(
            model, val_loader, bp_criterion, pulse_criterion, epoch
        )
        
        # Step the scheduler
        scheduler.step(val_loss)
        
        # Log metrics
        if config.use_wandb:
            wandb.log({
                'epoch': epoch,
                'train/loss': train_loss,
                'train/bp_loss': train_bp_loss,
                'train/pulse_loss': train_pulse_loss,
                'val/loss': val_loss,
                'val/bp_loss': val_bp_loss,
                'val/pulse_loss': val_pulse_loss,
                'lr': optimizer.param_groups[0]['lr']
            })
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': val_loss,
            }, os.path.join(config.checkpoint_dir, 'best_model.pth'))
            print(f"Saved new best model with validation loss: {val_loss:.4f}")

# Start training
if __name__ == "__main__":
    main()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/peter/.netrc.


wandb: Currently logged in as: pohsuanh (pohsuanh-university-of-southern-california) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 [Train]:   0%|          | 0/36 [08:26<?, ?it/s]


KeyboardInterrupt: 

socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7afeeb246950>> (for post_run_cell), with arguments args (<ExecutionResult object at 7aff241472e0, execution_count=65 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7aff24145c00, raw_cell="# Training Execution
def main():
    # Initialize .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22736e617065227d/home/peter/NSF_Project/Distraction1/Benefit_of_Distraction-master1/get_Dyno/training/Train_LinearTransformer.ipynb#X21sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe